# PV-MPC-Auction --- Colab driver

Companion notebook for the ICST 2026 paper *"Blockchain-Anchored Sealed-Bid Auctions via Pedersen Commitments, Shamir Sharing, and Schnorr Zero-Knowledge Proofs"*.

This notebook walks through the protocol end-to-end and reproduces the scalability figure from the paper. It does **not** require an Ethereum testnet --- everything runs against the in-memory PoW chain simulator. To run the on-chain leg on Sepolia, see `solidity/deploy_and_run.py` in the repo and the `REMIX_GUIDE.md`.

**How to use this notebook:**
1. Click *Runtime > Run all* (or run cell-by-cell with Shift+Enter).
2. Cell 1 clones the GitHub repo and installs the package.
3. Cells 2-6 walk through each component (group, commitment, sharing, comparison, full protocol).
4. Cell 7 reproduces the scalability figure.
5. Cell 8 runs the pytest suite.

Expected runtime on a free Colab CPU: ~3 minutes.

## Cell 1 --- Install + clone repo

In [ ]:
import os, sys, subprocess

# Edit this line if you forked the repo to your own GitHub:
REPO_URL = "https://github.com/Sanjoy-Chattopadhay/pv-mpc-auction.git"
REPO_DIR = "/content/pv-mpc-auction"

if not os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])

os.chdir(REPO_DIR)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"])
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
print("Installed. Working dir:", os.getcwd())

## Cell 2 --- Group parameters

We use the RFC 3526 2048-bit MODP Group 14 in production, but switch to a smaller 256-bit safe-prime group for Colab so the demos run in seconds.

In [ ]:
from pv_mpc_auction.group import default_group, small_test_group

G = small_test_group()         # comment out and use default_group() for the full 2048-bit version
print(f"Using {G.name}")
print(f"  p bits: {G.p.bit_length()}")
print(f"  q bits: {G.q.bit_length()}")
print(f"  g     : {G.g}")
print(f"  h     : {hex(G.h)[:32]}...")

## Cell 3 --- Pedersen commitment + Schnorr ZKP

In [ ]:
from pv_mpc_auction import pedersen

bid = 42_000   # an example sealed bid
C, r = pedersen.commit(G, bid)
print(f"Commitment C  = {hex(C.value)[:40]}...")
print(f"Randomness r  = {hex(r)[:40]}...")

# Produce a non-interactive proof of knowledge
proof = pedersen.prove(C, bid, r)
print(f"\nProof = (tau, s_m, s_r)")
print(f"  tau = {hex(proof.tau)[:40]}...")
print(f"  s_m = {hex(proof.s_m)[:40]}...")
print(f"  s_r = {hex(proof.s_r)[:40]}...")

# Verify
print(f"\nVerify opening : {pedersen.verify_opening(C, bid, r)}")
print(f"Verify ZKP     : {pedersen.verify(C, proof)}")

# Soundness: tampering with the bid is detected
print(f"Tampered bid   : {pedersen.verify_opening(C, bid + 1, r)}")

## Cell 4 --- Shamir secret sharing

Threshold $t = 3$, total $n = 5$ shares. Any 3 reconstruct; fewer reveal nothing.

In [ ]:
from pv_mpc_auction import shamir

secret = 123_456_789
shares = shamir.split(secret, n=5, t=3, q=G.q)
for s in shares:
    print(f"  share[{s.i}] = {s.value % 10**12:>12}")

print(f"\nReconstruct from 3 shares: {shamir.reconstruct(shares[:3], q=G.q)}")
print(f"Reconstruct from 4 shares: {shamir.reconstruct(shares[:4], q=G.q)}")
print(f"Reconstruct from 5 shares: {shamir.reconstruct(shares, q=G.q)}")

# Demo: homomorphic linear combination
a = shamir.split(7, n=5, t=3, q=G.q)
b = shamir.split(11, n=5, t=3, q=G.q)
summed = shamir.add_shares(a, b, q=G.q)
print(f"\n[7] + [11] reconstructed = {shamir.reconstruct(summed[:3], q=G.q)}")

## Cell 5 --- Secure greater-than (SecGT) tournament

Eight bidders, all bids hidden in their bit-shares. The tournament finds the maximum without revealing any individual bid.

In [ ]:
from pv_mpc_auction import secgt

bids = [120, 45, 380, 75, 210, 90, 175, 55]
n = len(bids)
t = n // 2 + 1
L = 10                       # bit-length per bid
winner_idx, winning_bid = secgt.tournament_winner(
    bids, bit_length=L, n=n, t=t, q=G.q
)
print(f"Bidders        : {bids}")
print(f"Winner (1-idx) : P{winner_idx}")
print(f"Winning bid    : {winning_bid}")
assert winning_bid == max(bids)
print("OK -- tournament correctly identified the maximum.")

## Cell 6 --- Full five-phase auction protocol

All phases including the simulated PoW chain. Each phase ends with a mined block.

In [ ]:
from pv_mpc_auction.protocol import run_auction
import secrets

bids = [secrets.randbelow(1 << 12) for _ in range(8)]
print("Sealed bids (these would not be revealed in a real run):")
for i, b in enumerate(bids, 1):
    print(f"  P{i}: {b}")

result = run_auction(bids, group=G, bit_length=12, chain_difficulty=0)

print(f"\nWinner       : P{result.winner_index}")
print(f"Winning bid  : {result.winning_bid}")
print(f"Verified     : {result.verified}")
print(f"Total time   : {result.timings.total*1000:.1f} ms")
print(f"\nPer-phase breakdown (ms):")
for phase in ['registration', 'commitment', 'sharing', 'mpc', 'verification']:
    print(f"  {phase:<14} {getattr(result.timings, phase)*1000:>8.2f}")

print(f"\nChain length : {len(result.chain.blocks)} blocks (genesis + 5 phases)")
ok, msg = result.chain.verify()
print(f"Chain valid  : {ok}  ({msg})")

## Cell 7 --- Scalability sweep (the paper's main figure)

Runs the protocol for $n \in \{3, 5, 7, 10, 12, 15, 20\}$, three trials each, and reproduces `fig_auction_scalability.pdf`. ~2-3 minutes on Colab.

In [ ]:
from pathlib import Path
from pv_mpc_auction.benchmark import bench_scalability, plot_scalability, bench_primitives

out = Path("colab_figures")
out.mkdir(exist_ok=True)

print("Primitive micro-benchmarks:")
for k, v in bench_primitives(G, trials=30).items():
    print(f"  {k:<32} {v:>7.2f} ms")

print("\nScalability sweep (n = 3, 5, 7, 10, 12, 15, 20)...")
records = bench_scalability(
    G,
    n_values=[3, 5, 7, 10, 12, 15, 20],
    bit_length=12,
    trials=3,
)
plot_scalability(records, out / "fig_auction_scalability.pdf")

from IPython.display import Image, display
import matplotlib.pyplot as plt, matplotlib.image as mpimg

# Re-plot inline
import statistics
ns = sorted({r['n'] for r in records})
means = [statistics.mean(r['total_s']*1000 for r in records if r['n']==n) for n in ns]
plt.figure(figsize=(6,3.5))
plt.plot(ns, means, marker='o')
plt.xlabel('Number of bidders n')
plt.ylabel('End-to-end latency (ms)')
plt.title('PV-MPC-Auction scalability')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Cell 8 --- Test suite

Runs all 25+ pytest tests. All should pass.

In [ ]:
!pytest -v

## Cell 9 --- (Optional) Deploy to Sepolia

If you want to drive the real `MPCAuction.sol` contract on the Sepolia testnet from Colab, uncomment the cell below. You will need:

1. **An Alchemy/Infura HTTP RPC URL.** Sign up at https://alchemy.com (free) and create a Sepolia app.
2. **A funded Sepolia private key.** Create a MetaMask wallet (fresh, never used on mainnet), switch to Sepolia, and request 0.5 ETH from https://sepoliafaucet.com (or https://sepolia-faucet.pk910.de as a PoW backup).
3. **Paste both into the cell below.** They are stored only in this Colab session.

**Never use a key that has any mainnet balance. Make a fresh one.**

In [ ]:
# Uncomment and fill in to run on Sepolia:
#
# import os
# os.environ['SEPOLIA_RPC_URL']     = 'https://eth-sepolia.g.alchemy.com/v2/YOUR_KEY'
# os.environ['SEPOLIA_PRIVATE_KEY'] = '0xYOUR_64_HEX_PRIVATE_KEY'
# !cd solidity && python deploy_and_run.py --n 5